In [2]:
!pip install BERTopic

In [3]:
!pip install bertopic
!pip install --upgrade joblib==1.1.0 # HDBScan fails to load with joblib==1.2.0
!pip install --upgrade git+https://github.com/scikit-learn-contrib/hdbscan.git#egg=hdbscan

  Attempting uninstall: joblib
    Found existing installation: joblib 1.6.0
    Uninstalling joblib-1.6.0:
      Successfully uninstalled joblib-1.6.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires joblib>=1.4, but you have joblib 1.1.0 which is incompatible.
imbalanced-learn 0.14.2 requires joblib<2,>=1.2.0, but you have joblib 1.1.0 which is incompatible.
scikit-learn 1.6.1 requires joblib>=1.2.0, but you have joblib 1.1.0 which is incompatible.
  Cloning https://github.com/scikit-learn-contrib/hdbscan.git to /tmp/pip-install-kx8cvlln/hdbscan_8afd839e2d1f4df49a48a73c336d719a
  Running command git clone --filter=blob:none --quiet https://github.com/scikit-learn-contrib/hdbscan.git /tmp/pip-install-kx8cvlln/hdbscan_8afd839e2d1f4df49a48a73c336d719a
  Resolved https://github.com/scikit-learn-contrib/hdbscan.git to commit a70b7b7728159baa3

In [4]:

import os
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP # In case we want to make results deterministic and reproducible

In [7]:
with open("/content/drive/MyDrive/stopwords.txt", 'r', encoding="utf8") as f:
    stop = f.read().splitlines()

vectorizer_model = CountVectorizer(ngram_range=(1, 1), stop_words=frozenset(stop))

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# Prepare data
df = pd.read_csv('/content/drive/MyDrive/reviews_prep.csv', encoding="utf-8")
docs = df['clean_text'].dropna().astype(str).values.tolist()


In [9]:
df.head(5)

,text,label_text,label,id,clean_text
0,всё пришло спасибо. только немного короче чем ...,positive,2,64971,всё прийти спасибо немного короче ожидать всё ...
1,The color of the dress is not the same as in t...,neutral,1,34896,the color of the dress is not the same as in t...
2,"Заказывала белый цвет, пришёл какой то персико...",neutral,1,46253,заказывать белый цвет прийти персиковый ответи...
3,Приятное к телу. Но в бюсте на размер больше. ...,negative,0,19034,приятный тело бюст оформить январь прибыть январь
4,Не пришла деньги через спор вернули,neutral,1,36063,прийти спор вернуть


In [10]:
from bertopic import BERTopic
import pandas as pd
import nltk
nltk.download('stopwords')
#from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [11]:
df.head(5)

,text,label_text,label,id,clean_text
0,всё пришло спасибо. только немного короче чем ...,positive,2,64971,всё прийти спасибо немного короче ожидать всё ...
1,The color of the dress is not the same as in t...,neutral,1,34896,the color of the dress is not the same as in t...
2,"Заказывала белый цвет, пришёл какой то персико...",neutral,1,46253,заказывать белый цвет прийти персиковый ответи...
3,Приятное к телу. Но в бюсте на размер больше. ...,negative,0,19034,приятный тело бюст оформить январь прибыть январь
4,Не пришла деньги через спор вернули,neutral,1,36063,прийти спор вернуть


In [12]:
docs

['всё прийти спасибо немного короче ожидать всё супер',
 'the color of the dress is not the same as in the picture he is red with greenish podton did not like the color the quality of the dress is good',
 'заказывать белый цвет прийти персиковый ответить подождать пара день пропасть целое хороший никакой торчать нитка жаль прийти такой цвет',
 'приятный тело бюст оформить январь прибыть январь',
 'прийти спор вернуть',
 'чёрный красный чёрный идеальный красный тонкий лосиный связь это оказаться маленький висеть складка итог российский чёрный идеальный ткань красный чёрный разный абсолютно',
 'тряпка половой бесформенный тоненький мешок стоить свой деньга копейка',
 'оооочень долго идти скорее дело работа служба спсркапец давать соседний город везти месяц отпуск успеть вернутьсян пригодиться вещь жаль тк сам дело шорты клёвый российский xl свободно сидеть счёт ремень держатсяможный смело малый выбирать советовать спср доставка маленький город',
 'прийти быстро неделя хороший сшить нитка

In [13]:
from nltk.corpus import stopwords


In [14]:
# Remove stopwords **after** generating the clusters
# NOTE: You can also perform the lemmatization here
my_stopwords = stopwords.words("russian")
vectorizer_model = CountVectorizer(stop_words=my_stopwords)

In [15]:
#from hdbscan import HDBSCAN
documents = df['clean_text'].values.tolist()
umap_model = UMAP(n_neighbors=10, n_components=3,
                  min_dist=0.0, metric='cosine', random_state=13)

In [16]:


# Load KBLab's Swedish sentence transformer model
sentence_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

# Initialize BERTopic with the settings we want
topic_model = BERTopic(embedding_model=sentence_model,
                       vectorizer_model=vectorizer_model,
                       umap_model=umap_model,
                       calculate_probabilities=True,
                       verbose=True)

# Fit the model
#topics, probs = topic_model.fit_transform(documents)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

... (4 outputs left out)

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [17]:
documents = [str(doc) for doc in documents if pd.notna(doc) and str(doc).strip() != '']
# Fit the model
topics, probs = topic_model.fit_transform(documents)

2026-11-25 11:38:31,734 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

2026-11-25 11:38:35,544 - BERTopic - Embedding - Completed ✓
2026-11-25 11:38:35,545 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-11-25 11:38:53,772 - BERTopic - Dimensionality - Completed ✓
2026-11-25 11:38:53,774 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-25 11:38:53,923 - BERTopic - Cluster - Completed ✓
2026-11-25 11:38:53,928 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-25 11:38:53,991 - BERTopic - Representation - Completed ✓


In [18]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,515,-1_цвет_хороший_прийти_подойти,"[цвет, хороший, прийти, подойти, рост, заказыв...",[большой спасибо продавец петербург идти недел...
1,0,172,0_спор_вернуть_месяц_прийти,"[спор, вернуть, месяц, прийти, ждать, получить...","[прийти ждать месяц деньга товар, ждать месяц ..."
2,1,103,1_платье_юбка_носить_джинсы,"[платье, юбка, носить, джинсы, рост, цена, тал...",[платье ужасный фото тонкий тянуться сильно за...
3,2,81,2_прийти_описание_хл_соответствовать,"[прийти, описание, хл, соответствовать, присла...","[соответствовать описание, прийти вино продаве..."
4,3,60,3_русский_российский_москва_подойти,"[русский, российский, москва, подойти, рост, н...",[довольный частный приходить товар такой хорош...
5,4,40,4_нитка_торчать_шов_ткань,"[нитка, торчать, шов, ткань, рукав, хлопок, си...",[ткань соответствовать заявить описание ярлык ...
6,5,39,5_спасибо_продавец_большой_быстро,"[спасибо, продавец, большой, быстро, рекомендо...",[всё отлично ткань приятный тело соответствова...
7,6,37,6_тонкий_ткань_синтетик_мало,"[тонкий, ткань, синтетик, мало, коротковатый, ...","[низкий посадка ткань тонкий просвечивать, тон..."
8,7,35,7_спор_вернуть_открыть_прийти,"[спор, вернуть, открыть, прийти, открывать, ст...","[прийти вернуть спор, прийти открыть спор верн..."
9,8,33,8_рукав_короткий_мешок_длина,"[рукав, короткий, мешок, длина, маленький, узк...",[сесть стирка рукав короткий стать длина корот...


In [19]:
topic_model.get_topic(0)

[('спор', np.float64(0.04695583878133104)),
 ('вернуть', np.float64(0.04315701618952843)),
 ('месяц', np.float64(0.03527728969462803)),
 ('прийти', np.float64(0.031216636524430345)),
 ('ждать', np.float64(0.030684620436287564)),
 ('получить', np.float64(0.030232644112703027)),
 ('продавец', np.float64(0.026231298828362382)),
 ('открыть', np.float64(0.025333518134958488)),
 ('отправить', np.float64(0.02237353516334237)),
 ('товар', np.float64(0.02209460658095293))]

In [30]:
topic_model.get_representative_docs(0)

['прийти открыть спор вернуть', 'прийти спор вернуть', 'прийти вернуть спор']

In [20]:
topic_model.get_topic(4) # topic_model.get_topic(similar_topics[0])

[('нитка', np.float64(0.08228403815366267)),
 ('торчать', np.float64(0.07291149606034862)),
 ('шов', np.float64(0.059666232443650956)),
 ('ткань', np.float64(0.04697040247646108)),
 ('рукав', np.float64(0.03010677948461121)),
 ('хлопок', np.float64(0.029948577662423648)),
 ('сильно', np.float64(0.02719736809993484)),
 ('чёрный', np.float64(0.02636599126099224)),
 ('красный', np.float64(0.025832611501383426)),
 ('прошить', np.float64(0.025019158151429716))]

In [21]:
topic_model.visualize_topics()

In [22]:
fig = topic_model.visualize_topics()
fig.write_html("/content/reviews_topics.html")

In [23]:
topic_model.visualize_hierarchy()

In [24]:
hierarchical_topics = topic_model.hierarchical_topics(documents)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

100%|██████████| 28/28 [00:00<00:00, 295.08it/s]


In [25]:
topic_model.reduce_topics(docs, nr_topics=20)


2026-11-25 11:40:27,736 - BERTopic - Topic reduction - Reducing number of topics
2026-11-25 11:40:27,749 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-25 11:40:27,790 - BERTopic - Representation - Completed ✓
2026-11-25 11:40:27,792 - BERTopic - Topic reduction - Reduced number of topics from 30 to 20


In [26]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,515,-1_цвет_хороший_прийти_подойти,"[цвет, хороший, прийти, подойти, рост, заказыв...",[прийти день огонь трек отслеживаться куртка х...
1,0,207,0_спор_вернуть_прийти_открыть,"[спор, вернуть, прийти, открыть, месяц, получи...","[прийти открыть спор вернуть, прийти спор верн..."
2,1,166,1_платье_короткий_рукав_рост,"[платье, короткий, рукав, рост, носить, длина,...",[прийти платье дырка сзади сшить тонкий ткань ...
3,2,134,2_прийти_быстро_заказывать_день,"[прийти, быстро, заказывать, день, хл, довольн...","[прийти другой, прислать день понравиться зака..."
4,3,77,3_ткань_нитка_торчать_шов,"[ткань, нитка, торчать, шов, тонкий, синтетик,...","[торчать нитка белый цвет, торчать нитка расхо..."
5,4,67,4_спасибо_продавец_приятный_неделя,"[спасибо, продавец, приятный, неделя, мягкий, ...",[посылка прийти быстро два неделя томск халати...
6,5,60,5_русский_российский_москва_подойти,"[русский, российский, москва, подойти, рост, н...",[заказывать русский оказаться короткий рукав ш...
7,6,46,6_запах_синтетик_ужасный_стирка,"[запах, синтетик, ужасный, стирка, плохой, неп...",[everywhere the threads stick out and open mon...
8,7,32,7_футболка_грудь_размер_продавец,"[футболка, грудь, размер, продавец, рубашка, т...",[футболка подойти русый идеально смотреться кр...
9,8,28,8_куртка_пальто_хороший_синий,"[куртка, пальто, хороший, синий, голубой, мину...",[куртка хороший цвет соответствуетный картинка...


In [27]:
topic_model.get_topic(3)

[('ткань', np.float64(0.0859619712176933)),
 ('нитка', np.float64(0.0744988286593872)),
 ('торчать', np.float64(0.06276919161095473)),
 ('шов', np.float64(0.05803304836906998)),
 ('тонкий', np.float64(0.057379987201708976)),
 ('синтетик', np.float64(0.03741679334669934)),
 ('хлопок', np.float64(0.029542917183355846)),
 ('рукав', np.float64(0.02924564143703406)),
 ('чёрный', np.float64(0.02633372721220361)),
 ('просвечивать', np.float64(0.024038891395959126))]

In [28]:
topic_model.visualize_topics()


In [29]:
topic_model.visualize_barchart(width=280, height=330,top_n_topics=20, n_words=20)


In [33]:
fig4 = topic_model.visualize_barchart(width=280, height=330, top_n_topics=20, n_words=20)
fig4.write_html("/content/sample_data/reviews_topic_wordBars.html")

In [35]:
fig5 = topic_model.visualize_heatmap(n_clusters=9)
fig5.show()
fig5.write_html("/content/sample_data/reviews_topic_corr.html")